<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W12/Q2/ASSIGNMNT_W12_Q_2_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TASK 2: IMPLEMENTING WEAK ENTITY RELATIONSHIPS
# IIT Jodhpur - Introduction to Databases
# ============================================================

import os
import sqlite3
import zipfile
from google.colab import files

# Create output directory
OUTPUT_DIR = "/content/Task_2_Submission"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# FILE 1: SQL SCHEMA
# ============================================================

sql_content = """
-- TASK 2: IMPLEMENTING WEAK ENTITY RELATIONSHIPS

-- Create the strong entity EMPLOYEE
CREATE TABLE EMPLOYEE (
    Ssn VARCHAR(20) PRIMARY KEY
);

-- Create the weak entity DEPENDENT
CREATE TABLE DEPENDENT (
    Essn VARCHAR(20) NOT NULL,
    Dependent_name VARCHAR(100) NOT NULL,
    Sex VARCHAR(10),
    Birth_date DATE,
    Relationship VARCHAR(50),

    PRIMARY KEY (Essn, Dependent_name),

    FOREIGN KEY (Essn)
        REFERENCES EMPLOYEE(Ssn)
        ON DELETE CASCADE
);

-- Insert sample employee records
INSERT INTO EMPLOYEE (Ssn)
VALUES
('EMP001'),
('EMP002'),
('EMP003');

-- Insert sample dependent records
INSERT INTO DEPENDENT
(Essn, Dependent_name, Sex, Birth_date, Relationship)
VALUES
('EMP001', 'Rahul', 'Male', '2010-05-15', 'Son'),
('EMP001', 'Priya', 'Female', '2012-08-20', 'Daughter'),
('EMP002', 'Amit', 'Male', '2008-03-10', 'Son'),
('EMP003', 'Neha', 'Female', '2015-11-25', 'Daughter');

-- Display employee records
SELECT * FROM EMPLOYEE;

-- Display dependent records
SELECT * FROM DEPENDENT;

-- Display employees with their dependents
SELECT
    E.Ssn,
    D.Dependent_name,
    D.Sex,
    D.Birth_date,
    D.Relationship
FROM EMPLOYEE E
JOIN DEPENDENT D
ON E.Ssn = D.Essn;
"""

sql_path = os.path.join(OUTPUT_DIR, "dependent_schema.sql")

with open(sql_path, "w") as f:
    f.write(sql_content)


# ============================================================
# FILE 2: TEXT-BASED RELATIONAL SCHEMA
# ============================================================

schema_content = """
TASK 2: IMPLEMENTING WEAK ENTITY RELATIONSHIPS

1. EMPLOYEE (STRONG ENTITY)

EMPLOYEE (
    Ssn PRIMARY KEY
)

Primary Key:
Ssn


2. DEPENDENT (WEAK ENTITY)

DEPENDENT (
    Essn,
    Dependent_name,
    Sex,
    Birth_date,
    Relationship
)

Composite Primary Key:
(Essn, Dependent_name)

Foreign Key:
Essn REFERENCES EMPLOYEE(Ssn)


3. KEY ATTRIBUTES

Strong Entity Primary Key:
Ssn

Weak Entity Partial Key:
Dependent_name

Weak Entity Composite Primary Key:
(Essn, Dependent_name)


4. EXISTENCE DEPENDENCY

The DEPENDENT entity depends on EMPLOYEE for its existence.
Every dependent must be associated with an existing employee.

The Essn attribute establishes the identifying relationship
between EMPLOYEE and DEPENDENT.
"""

txt_path = os.path.join(OUTPUT_DIR, "dependent_schema.txt")

with open(txt_path, "w") as f:
    f.write(schema_content)


# ============================================================
# FILE 3: RELATIONSHIP DIAGRAM
# ============================================================

diagram_content = """
TASK 2: WEAK ENTITY RELATIONSHIP DIAGRAM

             EMPLOYEE
        +------------------+
        | Ssn (Primary Key)|
        +------------------+
                 |
                 |
          Identifying
          Relationship
                 |
                 |
              DEPENDENT
       +-----------------------+
       | Essn (Foreign Key)    |
       | Dependent_name        |
       | Sex                   |
       | Birth_date            |
       | Relationship          |
       +-----------------------+

COMPOSITE PRIMARY KEY:
(Essn, Dependent_name)


RELATIONSHIP DESCRIPTION:

EMPLOYEE (1) -------- (N) DEPENDENT

One employee can have multiple dependents.
Each dependent must belong to exactly one employee.

Essn in DEPENDENT references Ssn in EMPLOYEE.

If an employee is deleted, their dependent records
are automatically deleted using ON DELETE CASCADE.
"""

diagram_path = os.path.join(
    OUTPUT_DIR, "weak_entity_relationship.md"
)

with open(diagram_path, "w") as f:
    f.write(diagram_content)


# ============================================================
# FILE 4: PRIMARY KEY JUSTIFICATION REPORT
# Maximum 100 words
# ============================================================

report_content = """
PRIMARY KEY JUSTIFICATION REPORT

The DEPENDENT table uses a composite primary key consisting
of Essn and Dependent_name. Essn identifies the employee
who owns the dependent, while Dependent_name acts as the
partial key of the weak entity. Together, they uniquely
identify each dependent. Essn also serves as a foreign key
referencing EMPLOYEE(Ssn), establishing referential
integrity and existence dependency. This design allows
different employees to have dependents with identical
names while preventing duplicate dependent names for
the same employee.
"""

report_path = os.path.join(
    OUTPUT_DIR, "dependent_primary_key_report.md"
)

with open(report_path, "w") as f:
    f.write(report_content)


# ============================================================
# FILE 5: EXECUTE AND VALIDATE SQL USING SQLITE
# ============================================================

db_path = os.path.join(OUTPUT_DIR, "dependent_database.db")

conn = sqlite3.connect(db_path)

# Enable foreign key enforcement
conn.execute("PRAGMA foreign_keys = ON")

# Execute SQL statements
conn.executescript(sql_content)

# Verify EMPLOYEE table
print("\nEMPLOYEE TABLE:")
for row in conn.execute("SELECT * FROM EMPLOYEE"):
    print(row)

# Verify DEPENDENT table
print("\nDEPENDENT TABLE:")
for row in conn.execute("SELECT * FROM DEPENDENT"):
    print(row)

# Verify relationship
print("\nEMPLOYEE - DEPENDENT RELATIONSHIP:")

for row in conn.execute("""
    SELECT E.Ssn, D.Dependent_name,
           D.Sex, D.Birth_date, D.Relationship
    FROM EMPLOYEE E
    JOIN DEPENDENT D
    ON E.Ssn = D.Essn
"""):
    print(row)

# Check foreign key constraints
print("\nFOREIGN KEY VALIDATION:")
print(list(conn.execute("PRAGMA foreign_key_check")))

conn.close()


# ============================================================
# FILE 6: CREATE ZIP PACKAGE
# ============================================================

zip_path = "/content/Task_2_Database_Assignment.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    for filename in os.listdir(OUTPUT_DIR):
        filepath = os.path.join(OUTPUT_DIR, filename)

        if os.path.isfile(filepath):
            zipf.write(filepath, arcname=filename)


# ============================================================
# DISPLAY GENERATED FILES
# ============================================================

print("\n" + "=" * 55)
print("TASK 2 - FILES GENERATED SUCCESSFULLY")
print("=" * 55)

for filename in os.listdir(OUTPUT_DIR):
    print("SUCCESS:", filename)

print("\nZIP PACKAGE CREATED:")
print(zip_path)

# Automatically download ZIP
files.download(zip_path)